In [1]:
"""
The primary data is organized into a dictionary based on the mc2d-id and subsections.
Each subsection can have a 'citations' field containing a list of citations

{
  "mc2d-10": {
    "general": {...},
    "parents_3d": {...},
    "electronic": {...},
    "vibrational": {...},
    "example": {...}
  }
}
"""

import json
import copy
import numpy as np

from pathlib import Path

from pymatgen.core import Composition

import aiida
from aiida import orm
aiida.load_profile("mc2d-new")


Profile<uuid='dee363765806401c8ea7e3b50e15bee2' name='mc2d-new'>

In [2]:
SOURCE_JSON = Path("../../../../source-data/mc2d-data.json")

with open(SOURCE_JSON) as fh:
    SOURCE_DATA = json.load(fh)

ID_JSON = Path("../../../../ids/mc2d-ids.json")

with open(ID_JSON) as fh:
    ID_DATA = json.load(fh)

ids_reversed = {opt_uuid: id for id, opt_uuid in ID_DATA.items()}

In [3]:
def convert_binding_energy_to_mev(parents):
    """
    Binding energy is in eV/ang^2 in the original source data.
    Convert to meV/ang^2 for Materials Cloud discover section.
    """
    converted_parents = []
    for parent in parents:
        cparent = copy.deepcopy(parent)
        for be_key in ["binding_energy_df2", "binding_energy_rvv10"]:
            be_entry = cparent.get(be_key)
            if be_entry:
                if be_entry.get('value') is not None:
                    cparent[be_key]['value'] *= 1000
        converted_parents.append(cparent)
    return converted_parents

def get_pymatgen_structure(aiida_structure_node):
    c = copy.deepcopy(aiida_structure_node)
    c.pbc = (True, True, True)
    return c.get_pymatgen_structure()

def get_formulas(structure_uuid):
    structure_node = orm.load_node(structure_uuid)
    # pymatgen iupac formula
    pmg_formula = get_pymatgen_structure(structure_node).composition.reduced_formula
    # full hill formula:
    hill_formula = structure_node.get_formula(mode="hill")

    return pmg_formula, hill_formula

def pmg_convert_prototype_to_iupac(formula):
    if formula:
        try:
            return Composition(formula).reduced_formula
        except ValueError:
            # fails for cases such as P(black). just keep original formula
            return formula
    else:
        return None

In [4]:
def check_if_phonons_unstable(phonon_bands_node):
    THRESHOLD = -0.3 # unit THz and equivalently -10 cm-1
    return bool(np.min(phonon_bands_node.get_bands()) < THRESHOLD)

def get_vibrational_section(entry):
    vibrational = {}
    phonon_bands_uuid = entry.get("phonon_bands_uuid")
    if phonon_bands_uuid:
        phonon_bands_node = orm.load_node(phonon_bands_uuid)
        if isinstance(phonon_bands_node, orm.BandsData):
            vibrational["phonon_bands_uuid"] = phonon_bands_uuid
            vibrational["unstable"] = check_if_phonons_unstable(phonon_bands_node)
        else:
            print(f"Warning: {phonon_bands_uuid} is not BandsData, skipping")
    return vibrational


In [5]:
formatted_data = {}

for entry in SOURCE_DATA:

    mc2d_id = f"mc2d-{ids_reversed[entry['optimized_2D_structure_uuid']]}"

    pmg_formula, hill_formula = get_formulas(entry['optimized_2D_structure_uuid'])

    formatted_data[mc2d_id] = {
        "general": {
            "formula": pmg_formula,
            "formula_hill": hill_formula,
            "space_group_number": entry.get("space_group_number"),
            "prototype": pmg_convert_prototype_to_iupac(entry.get("prototype")),
            "abundance": entry.get("abundance"),
            "structure_relaxed_uuid": entry.get("optimized_2D_structure_uuid"),
            "structure_extracted_uuid": entry.get("as_extracted_2D_structure_uuid"),
            "citations": entry.get("citations"),
        },
        "parents_3d": convert_binding_energy_to_mev(entry.get("all_3D_parents")),
        "electronic": {
            "bands_uuid": entry.get("bands_uuid"),
            "band_gap": entry.get("band_gap"),
            "fermi_energy": entry.get("fermi_energy"),
            "magnetic_state": entry.get("magnetic_state"),
            "absolute_magnetization": entry.get("absolute_magnetization"),
            "total_magnetization": entry.get("total_magnetization"),
        },
        "vibrational": get_vibrational_section(entry)
    }

In [6]:
OUT_JSON = Path("../base.json")
with OUT_JSON.open("w") as fh:
    json.dump(formatted_data, fh, indent=2)